<div style="text-align: center;">

  <img src="img/LogoUesc.png" alt="UESC" style="width: 60px; display: block; margin: 0 auto 5px auto;">

  <div style="font-size: 20px;"><strong>UNIVERSIDADE ESTADUAL DE SANTA CRUZ - UESC</strong></div>
  <div style="font-size: 18px;">DEPARTAMENTO DE ENGENHARIAS E COMPUTAÇÃO - DEC</div>
  <div style="font-size: 18px;">Grupo de Pesquisas em Simulações e Controle de Processos</div>

  <hr style="margin: 20px 0; border: 1px solid #0074B7;">

  <div style="font-size: 22px; font-weight: bold; margin-bottom: 10px;">PROGRAMAÇÃO EM PYTHON PARA AS ENGENHARIAS</div>

  <div style="font-size: 18px; margin-top: 10px;"><strong>Tema</strong>: Introdução a Redes Neurais Artificiais - Básico</div>
  <div style="font-size: 18px;"><strong>Prof. Dr. E.R.Edwards</strong></div>

  <div style="font-size: 16px; margin-top: 15px;">Ilhéus - BA, Abril de 2025</div>
  
  <hr style="margin: 20px 0; border: 1px solid #0074B7;">

</div>

**__Sumário__**

- [1. Problema de engenharia e dados experimentais](#secao-1)
- [2. Rede neural com quatro neurônios ocultos](#secao-2)
- [3. Modelo simplificado com entradas normalizadas](#secao-3)
- [4. Interpretação dos resultados e limitações](#secao-4)
- [5. Cálculo manual de uma previsão](#secao-5)
- [6. Verificação dos cálculos em Python](#secao-6)

<div id="secao-1" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">1. Problema de engenharia e dados experimentais</div>

Em um processo industrial de conversão de etileno ($C_2H_4$) em óxido de etileno ($C_2H_4O$), a eficiência do reator catalítico depende da __temperatura__ de operação e da __concentração__ inicial de etileno na corrente de entrada.

Com base em dados experimentais obtidos em planta piloto, deseja-se prever a conversão ($X$) da reação (em fração, de $0$ a $1$) a partir da Temperatura $T$ (em $^oC$) e da Concentração inicial $C_o$ (em mol/L). Para isso, uma __Rede Neural__ com uma camada oculta e __Função de Ativação Sigmoide__ será utilizada.

<img src="img/Tab_05.png" alt="UESC" style="width: 600px; display: block; margin: 0 auto 5px auto;">
<p style="text-align: center;"><strong>Figura: Tabela de dados experimentais (T, Co) e respectivas conversões (X).</strong></p>

Deseja-se:

1. Implementar uma rede neural simples para prever $X$ com base em $T$ e $C_o$;
2. Treinar a rede por $1000$ épocas;
3. Gerar gráfico de perda durante o treinamento;
4. Realizar previsões para novas condições operacionais;
5. Interpretar os resultados.

**Observação de engenharia.** Os cinco pares de dados constituem um exemplo didático. Temperatura e concentração variam simultaneamente, de modo que esses dados não permitem identificar separadamente a influência de cada variável. As previsões não devem ser utilizadas para projeto ou operação industrial sem validação experimental independente.

<div id="secao-2" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">2. Rede neural com quatro neurônios ocultos</div>

A primeira implementação utiliza duas entradas (temperatura e concentração), quatro neurônios ocultos com ativação sigmoide e um neurônio de saída. As entradas são normalizadas por min–max, utilizando somente os valores do conjunto de treinamento. A regra da cadeia é aplicada ao gradiente da camada oculta, incluindo os pesos da camada de saída.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

class NeuralNetwork:
    def __init__(self, seed=42):
        rng = np.random.default_rng(seed)
        self.w = rng.normal(scale=0.5, size=(4, 2))
        self.b = np.zeros(4)
        self.w_out = rng.normal(scale=0.5, size=4)
        self.b_out = 0.0
        self.loss_history = []

    def feedforward(self, x):
        z_h = self.w @ x + self.b
        h = sigmoid(z_h)
        z_o = self.w_out @ h + self.b_out
        y_hat = float(sigmoid(z_o))
        return y_hat, h

    def train(self, data, targets, epochs=1000, learn_rate=0.1):
        for epoch in range(epochs):
            for x, y in zip(data, targets):
                y_hat, h = self.feedforward(x)
                delta_out = 2.0 * (y_hat - y) * y_hat * (1.0 - y_hat)
                # Gradiente oculto calculado ANTES de atualizar w_out
                delta_h = delta_out * self.w_out * h * (1.0 - h)
                grad_w_out = delta_out * h
                grad_w = np.outer(delta_h, x)
                self.w_out -= learn_rate * grad_w_out
                self.b_out -= learn_rate * delta_out
                self.w -= learn_rate * grad_w
                self.b -= learn_rate * delta_h
            preds = np.array([self.feedforward(x)[0] for x in data])
            loss = np.mean((targets - preds)**2)
            self.loss_history.append(float(loss))
            if epoch in (0, 99, 199, 499, 999):
                print(f"Época {epoch + 1:4d} | MSE = {loss:.6f}")

    def predict(self, x):
        return self.feedforward(np.asarray(x, dtype=float))[0]

    def plot_loss(self):
        plt.figure(figsize=(8, 4.5))
        plt.plot(np.arange(1, len(self.loss_history) + 1), self.loss_history)
        plt.xlabel("Épocas")
        plt.ylabel("Erro quadrático médio (MSE)")
        plt.title("Evolução do erro de treinamento")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

**Dados experimentais, normalização e treinamento:**

In [ ]:
dados = np.array([[200, 1.0], [250, 1.2], [300, 1.5],
                  [350, 1.8], [400, 2.0]], dtype=float)
conversoes = np.array([0.10, 0.35, 0.60, 0.80, 0.95])

entrada_min = dados.min(axis=0)
entrada_max = dados.max(axis=0)
dados_norm = (dados - entrada_min) / (entrada_max - entrada_min)

nn = NeuralNetwork(seed=42)
nn.train(dados_norm, conversoes, epochs=1000, learn_rate=0.1)
nn.plot_loss()

**Previsões para condições intermediárias:** a mesma transformação min–max utilizada no treinamento deve ser aplicada às novas entradas.

In [ ]:
novos_dados = np.array([[275, 1.4], [375, 1.9],
                        [220, 1.1], [310, 1.6]], dtype=float)
novos_norm = (novos_dados - entrada_min) / (entrada_max - entrada_min)
for (T, C0), entrada in zip(novos_dados, novos_norm):
    previsao = nn.predict(entrada)
    print(f"T = {T:.0f} °C, C₀ = {C0:.2f} mol/L → X ≈ {previsao:.3f}")

<div id="secao-3" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">3. Modelo simplificado com entradas normalizadas</div>

Para comparar arquiteturas, consideramos também um **modelo sem camada oculta**, com uma combinação linear das duas entradas seguida de sigmoide. Esse modelo é diferente da rede anterior: ele possui menor capacidade de representar relações não lineares. O gradiente do MSE contém o fator 2 decorrente da derivação do erro quadrático.

In [ ]:
# Modelo alternativo: neurônio único com ativação sigmoide
X = dados_norm.copy()
y = conversoes.reshape(-1, 1)
rng = np.random.default_rng(0)
w = rng.normal(size=(2, 1))
b = 0.0
alpha = 0.1
erros = []

for epoca in range(1000):
    z = X @ w + b
    y_pred = sigmoid(z)
    erro = y_pred - y
    erros.append(float(np.mean(erro**2)))
    delta = 2.0 * erro * y_pred * (1.0 - y_pred)
    grad_w = X.T @ delta / len(X)
    grad_b = float(np.mean(delta))
    w -= alpha * grad_w
    b -= alpha * grad_b

plt.figure(figsize=(8, 4.5))
plt.plot(np.arange(1, 1001), erros)
plt.title("MSE do modelo com um neurônio")
plt.xlabel("Épocas")
plt.ylabel("Erro quadrático médio (MSE)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
condicoes = np.array([[275, 1.3], [375, 1.9]], dtype=float)
condicoes_norm = (condicoes - entrada_min) / (entrada_max - entrada_min)
for (T, C0), x_norm in zip(condicoes, condicoes_norm):
    pred = float(sigmoid(x_norm @ w + b))
    print(f"T = {T:.0f} °C, C₀ = {C0:.2f} mol/L → X ≈ {pred:.3f}")

<div id="secao-4" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">4. Interpretação dos resultados e limitações</div>

A redução do erro de treinamento indica que o modelo ajustou suas previsões aos cinco exemplos fornecidos, **mas não comprova sua capacidade de generalização**. O formato da curva de perda depende dos pesos iniciais, da normalização, da taxa de aprendizagem e da arquitetura; não é correto presumir que a função de custo de uma rede neural seja convexa.

A sigmoide limita a previsão ao intervalo entre 0 e 1, compatível com a definição de conversão em fração. Entretanto, essa restrição matemática não garante coerência físico-química em condições não observadas. Com apenas cinco dados e temperatura e concentração variando juntas, não é possível avaliar confiavelmente a influência isolada de cada variável nem a qualidade das extrapolações.

O modelo com quatro neurônios ocultos e o modelo com um neurônio são **duas abordagens diferentes**; seus pesos, curvas de erro e previsões não devem ser confundidos. Para aplicação real, seriam necessários mais dados, experimentos que variem as entradas de forma independente e avaliação com dados não utilizados no treinamento.

<div id="secao-5" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">5. Cálculo manual de uma previsão</div>

Para compreender o cálculo de propagação direta (*feedforward*), adotaremos um **conjunto ilustrativo de pesos e vieses fixos**, que **não corresponde necessariamente aos pesos obtidos nos treinamentos anteriores**. Nesta demonstração, a temperatura é inserida diretamente em graus Celsius, sem normalização, para reproduzir o cálculo numérico proposto. Essa escolha causa saturação de neurônios e **não deve ser interpretada como uma configuração recomendada para treinamento**.

Considere $T=325\ ^\circ\mathrm{C}$ e $C_0=1{,}6\ \mathrm{mol/L}$. A saída de cada neurônio oculto é

$$z_i=w_{i1}T+w_{i2}C_0+b_i,\qquad h_i=\sigma(z_i),\qquad \sigma(z)=\frac{1}{1+e^{-z}}.$$

Com os pesos e vieses apresentados no código da seção seguinte, obtêm-se:

$$\begin{aligned}
z_1&=-0{,}414(325)-0{,}141(1{,}6)-0{,}322=-135{,}0976,\\
z_2&=-1{,}572(325)+1{,}752(1{,}6)+0{,}584=-507{,}5128,\\
z_3&=0{,}702(325)-0{,}667(1{,}6)+0{,}166=227{,}2488,\\
z_4&=0{,}571(325)+1{,}428(1{,}6)-0{,}738=187{,}1218.
\end{aligned}$$

As ativações correspondentes são aproximadamente

$$h_1\approx 2{,}13\times10^{-59},\quad h_2\approx 3{,}89\times10^{-221},\quad h_3\approx1,\quad h_4\approx1.$$

Na saída:

$$z_{\mathrm{out}}=0{,}266h_1-2{,}305h_2-0{,}011h_3-0{,}550h_4+0{,}870\approx0{,}309.$$

Logo,

$$\boxed{\widehat X=\sigma(0{,}309)\approx0{,}5766=57{,}66\%}.$$

O resultado demonstra o cálculo de uma rede com pesos fixos; **não constitui validação da conversão de um reator real**.

<div id="secao-6" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">6. Verificação dos cálculos em Python</div>

In [ ]:
# Verificação independente dos cálculos manuais com pesos ilustrativos
import numpy as np

def sigmoid_estavel(z):
    z = np.asarray(z, dtype=float)
    return np.where(z >= 0,
                    1.0 / (1.0 + np.exp(-np.maximum(z, 0))),
                    np.exp(np.minimum(z, 0)) / (1.0 + np.exp(np.minimum(z, 0))))

entrada = np.array([325.0, 1.6])
pesos_ocultos = np.array([[-0.414, -0.141],
                          [-1.572,  1.752],
                          [ 0.702, -0.667],
                          [ 0.571,  1.428]])
bias_ocultos = np.array([-0.322, 0.584, 0.166, -0.738])
pesos_saida = np.array([0.266, -2.305, -0.011, -0.550])
bias_saida = 0.870

z_ocultos = pesos_ocultos @ entrada + bias_ocultos
h_ocultos = sigmoid_estavel(z_ocultos)
z_saida = pesos_saida @ h_ocultos + bias_saida
saida_final = float(sigmoid_estavel(z_saida))

print("z_ocultos =", z_ocultos)
print("h_ocultos =", h_ocultos)
print("z_saida =", z_saida)
print(f"Conversão ilustrativa X = {saida_final:.4f} ({100*saida_final:.2f}%)")

A implementação reproduz os cálculos apresentados na seção anterior. Observe que o uso de entradas não normalizadas e de pesos fixos é exclusivo desta demonstração algébrica. Para previsões com os modelos treinados, devem ser utilizadas as entradas normalizadas e os pesos correspondentes a cada modelo.

<div style="text-align: center; font-size: 12px; color: gray;
            margin-top: 40px; padding-top: 15px;
            border-top: 1px solid #dddddd;">

  Material desenvolvido pelo Grupo de Pesquisas em
  Modelagem Computacional da UESC.<br>

  © 2026 — Todos os direitos reservados.
  Uso comercial somente mediante autorização.

</div>